# Final Project
#### DACSS 756: Machine Learning for Social Scientists
#### 2026-05-13
#### Isabella Vallejo

---

# Introduction & Motivation

## Overview

- **Problem:** predict outcomes of drug-related ER visits from hospital admissions data.
- **Problem Type:** classification
- **Evaluation Metric:** recall (i.e., true positive rate)
- **Target Variable:** patient health outcome (binary, "positive" or "negative")
- **Predictor Variables:** substance use variables, hospital visit type, demographic variables, and temporal variables

## Summary

This project employs machine learning methods to predict outcomes of drug-related ER visits. I use the 2011 data from the Drug Abuse and Warning Network (DAWN), aggregated by SAMHSA. DAWN was initially created to function as a national health surveillance system in the United States that would help to track patterns and changes in drug use.

The DAWN dataset includes information abstracted from approximately 230k patient health records from drug-related ER visits. Information included in the dataset includes details about patient demographics, case type (e.g., suicide attempt, drug overdose, accidental ingestion...), patient outcome (e.g., died, admitted to hospital, discharged...), and substances involved in the visit. I use these data to build and evaluate three types of machine learning classifiers -- a logistic regression model with L1 regularization, a random forest classifier, and a support vector classifier -- to predict whether the visit outcome was positive (e.g., patient was discharged) or negative (e.g., patient died).

The aim of this project was to develop a classifier that could, in theory and with further refinement, serve a function like aiding medical staff with triaging drug-related ER visits in order to best allocate resources to the most high-risk patients. Implementation of this sort of method could potentially minimize negative patient outcomes by simplifying a time-consuming process for medical staff and accurately identifying high-risk patients that staff might overlook. The model may turn out to make predictions about outcomes that are more accurate than those a human might make as it has access to more information than any human can reasonably be expected to have (let alone utilize) in their decision-making process, and it will almost certainly make its predictions faster than a human. This project therefore has implications for improvement of health outcomes and the efficiency and quality of healthcare provision as a whole.

It is highly unlikely that we would ever want a machine learning model to make decisions about patient care on its own, but a model of this sort could function to simplify the process by flagging high-risk cases for future review. However, it is also important to note that use of a computational risk-assessment method could result in overreliance on it by medical staff; systems like this should be seen as an additional tool and not take the place of human evaluation and judgment. Additionally, there are other types of health information that were not collected by DAWN (e.g., past medical history) that would likely be useful for maximizing prediction accuracy; therefore, a more comprehensive approach would include information like this in the model training data.

# Setup

Install and import required libraries and modules:

In [ ]:
# install libraries
!pip install itables

In [ ]:
# import libraries and modules
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itables import init_notebook_mode
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import recall_score

Set notebook-wide settings:

In [ ]:
# set notebook-wide settings
np.set_printoptions(suppress = True)
init_notebook_mode(all_interactive = True)

# Data Cleaning

Import the full DAWN data, then take a random sample to reduce computational demands:

In [ ]:
# import full data
dawn_full = pd.read_csv(
    "/kaggle/input/datasets/bellavallejo/dawn-2011-samhda/DAWN_2011_SAMHDA.tsv", 
    sep = "\t"
)
# take a random sample of the data
dawn_sample = dawn_full.sample(
    axis = 'index', 
    frac = 0.1, 
    random_state = 1109
)

Lowercase column names and rename colummn `disposition` to `outcome`:

In [ ]:
# lowercase column names
dawn_sample.columns = dawn_sample.columns.str.lower()
# rename `disposition` to `outcome`
dawn_sample.rename(
    columns = {'disposition': 'outcome'}, 
    inplace = True
)

Select variables to keep, which include: visit outcome; patient age; patient sex; patient race, quarter of year in which visit occurred; part of day in which visit occurred; metropolitan area in which the hospital is located; case type; and whether or not alcohol, illicit drugs, pharmaceutical drugs, non-medical use of pharmaceutical drugs, and substance misuse were involved in the visit.

In [ ]:
# select variables to keep, except for `drugid`, `route`, and `toxtest` variables
colnames_vars = [
    'caseid', 'outcome', 'agecat', 'sex', 'race', 'quarter', 'daypart', 'metro', 'casetype', 'alcohol', 'nonalcill', 'pharma', 'nonmedpharma', 'allabuse', 'numsubs'
]

# create empty lists to store column names for `drugid`, `route`, and `toxtest` variables
colnames_drugid = []
colnames_route = []
colnames_toxtest = []

# create lists of all column names that record `drugid`, `route`, and `toxtest` variables
for i in range(1, 23, 1):
  colnames_drugid.append(f"drugid_{i}")
  colnames_route.append(f"route_{i}")
  colnames_toxtest.append(f"toxtest_{i}")

Combine all variables into one dataframe:

In [ ]:
# combine all selected variables into one dataframe
dawn_allvars = pd.concat([dawn_sample[colnames_vars], dawn_sample[colnames_drugid], dawn_sample[colnames_route], dawn_sample[colnames_toxtest]], axis = 'columns')

Recode missing values as `NA`:

In [ ]:
# recode missing values across the dataset
missing = [-9, -8, -7]
for value in missing:
  dawn_allvars.replace(
      {value: pd.NA}, 
      inplace = True
  )

Recode:

- `output`, to make `0` = positive visit outcome and `1` = negative visit outcome
- `route`, to make `98` ("multiple routes of administration") align with the values for other routes (i.e., `1` = oral, `2` = injected...)
- `toxtest`, to make `1` = presence of reported substance confirmed by toxicology and `0` = presence of reported substance not confirmed by toxicology

In [ ]:
# hide warnings from printed output
warnings.filterwarnings("ignore")

# recode `outcome`
dawn_allvars['outcome'].replace(
    {
        1: 0, 2: 0, 3: 0, 
        4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 
        96: pd.NA, -8: pd.NA
    }, 
    inplace = True
)

# recode `route`
dawn_allvars[colnames_route].replace(
    {-98: 7}, 
    inplace = True
)

# recode `toxtest`
dawn_allvars[colnames_toxtest].replace(
    {
        1: 1, 
        2: 0, 
        -9: pd.NA, -7: pd.NA
    }, 
    inplace = True
)

Drop missing values of target variable `outcome`:

In [ ]:
# drop missing values of `outcome`
dawn_allvars.dropna(
    subset = ['outcome'], 
    inplace = True
)

Dummy-code the variables that record drug ID numbers, routes of administration, and toxicology test results, so that there is one dummy-coded column for each possible value (e.g., `drugid_1` records `1` if the drug with ID #1 was involved in the case and `0` if it was not, `route_1` records `1` if the route of administration coded as `1` was involved in the case and `1` if it was not, and `toxtest` records `1` if toxicology testing confirmed the presence of *any* of the drugs involved in the case and `0` if it did not):

In [ ]:
# hide warnings from printed output
warnings.filterwarnings("ignore")

# create blank dataframe to store dummy variables in
dummies = pd.DataFrame()

# create dummy variables for `drugid` columns
values = range(1, 2559, 1)
for i in values:
  dummies[f"drugid_{i}"] = dawn_allvars[colnames_drugid].eq(i).any(axis = 'columns').astype(int)

# create dummy variables for `route` columns
values = range(1, 8, 1)
for i in values:
  dummies[f"route_{i}"] = dawn_allvars[colnames_route].eq(i).any(axis = 'columns').astype(int)

# create dummy variable for `toxtest` columns
dummies["toxtest"] = dawn_allvars[colnames_toxtest].eq(1).any(axis = 'columns').astype(int)

Add dummy variables to the data:

In [ ]:
# add dummy variables to the data
dawn = pd.concat(
    [
        dawn_allvars[colnames_vars], 
        dummies
    ], 
    axis = 'columns'
)

Remove any drugs from the data that were involved in less than 1% of cases:

In [ ]:
# drop any drugs that appear in less than 1% of the sample
not_drugs = dawn[colnames_vars + ['route_1', 'route_2', 'route_3', 'route_4', 'route_5', 'route_6', 'route_7', 'toxtest']]
drugs = dawn.drop(not_drugs, axis = 'columns')
common_drugs = drugs[drugs.columns[drugs.sum() / len(drugs) >= 0.01]]
uncommon_drugs = drugs.drop(common_drugs, axis = 'columns')
dawn.drop(uncommon_drugs, axis = 'columns', inplace = True)

 Give the remaining `drugid` variables descriptive names:

In [ ]:
# give `drugid` variables descriptive names
dawn.rename(
    columns = {
        'drugid_14': 'drug_ibuprofen',
        'drugid_21': 'drug_warfarin',
        'drugid_48': 'drug_acetaminophen',
        'drugid_49': 'drug_methadone',
        'drugid_85': 'drug_amoxicillin',
        'drugid_120': 'drug_sulfamethoxazole_trimethoprim',
        'drugid_140': 'drug_lorazepam',
        'drugid_152': 'drug_alprazolam',
        'drugid_154': 'drug_aspirin',
        'drugid_179': 'drug_clonazepam',
        'drugid_234': 'drug_ipratropium',
        'drugid_285': 'drug_oxycodone',
        'drugid_461': 'drug_lisinopril',
        'drugid_503': 'drug_amphetamine',
        'drugid_505': 'drug_methamphetamine',
        'drugid_553': 'drug_zolpidem',
        'drugid_865': 'drug_alcohol',
        'drugid_1016': 'drug_acetaminophen_hydrocodone',
        'drugid_1018': 'drug_acetaminophen_oxycodone',
        'drugid_1253': 'drug_heroin',
        'drugid_1254': 'drug_cocaine',
        'drugid_1255': 'drug_marijuana',
        'drugid_1451': 'drug_quetiapine',
        'drugid_2343': 'drug_narcotic_analgesics_nos',
        'drugid_2349': 'drug_benzodiazepines_nos',
        'drugid_2420': 'drug_drug_unknown',
        'drugid_2426': 'drug_poly_drugs',
        'drugid_2427': 'drug_antineoplastics_nos'
    }, 
    inplace = True
)

# Exploratory Data Analysis

Review the data:

In [ ]:
# review the clean data
print("Number of Rows:", len(dawn))
print("Number of Columns:", len(dawn.columns))
print("Number of Features:", len(dawn.drop(['caseid', 'outcome'], axis = 'columns').columns), "(excluding `caseid` and `outcome`)")

Target Variable:

`outcome`

Numeric Columns:

`caseid`, `numsubs`

Categorical Columns:

`outcome`, `agecat`, `sex`, `race`, `quarter`, `daypart`, `metro`, `casetype`, `alcohol`, `nonalcill`, `pharma`, `nonmedpharma`, `allabuse`, `route_1`, `route_2`, `route_3`, `route_4`, `route_5`, `route_6`, `route_7`, `toxtest`, `drug_ibuprofen`, `drug_warfarin`, `drug_acetaminophen`, `drug_methadone`, `drug_amoxicillin`, `drug_sulfamethoxazole_trimethoprim`, `drug_lorazepam`, `drug_alprazolam`, `drug_aspirin`, `drug_clonazepam`, `drug_ipratropium`, `drug_oxycodone`, `drug_lisinopril`, `drug_amphetamine`, `drug_methamphetamine`, `drug_zolpidem`, `drug_alcohol`, `drug_acetaminophen_hydrocodone`, `drug_acetaminophen_oxycodone`, `drug_heroin`, `drug_cocaine`, `drug_marijuana`,`drug_narcotic_analgesics_nos`, `drug_benzodiazepines_nos`, `drug_drug_unknown`, `drug_poly_drugs`, `drug_antineoplastics_nos`

Visualize distribution of the target variable `outcome` and the frequencies of occurrences of the drugs included in the sample:

In [ ]:
# visualize `outcome`
outcome_counts = dawn['outcome'].value_counts()
fig, ax = plt.subplots()
plt.bar(outcome_counts.index, outcome_counts.values)
plt.title("Distribution of Patient Outcomes")
plt.xticks(
    ticks = [0, 1] , 
    labels = ['Positive', 'Negative']
)
plt.xlabel("Outcome")
plt.ylabel("Number of Cases")
plt.show()

This plot indicates that the target variable `outcome` is slightly imbalanced (approximately a 2:1 ratio of positive:negative outcomes).

In [ ]:
# visualize drug variables
drug_cols = ['drug_ibuprofen', 'drug_warfarin', 'drug_acetaminophen', 'drug_methadone', 'drug_amoxicillin', 'drug_sulfamethoxazole_trimethoprim', 'drug_lorazepam', 'drug_alprazolam', 'drug_aspirin', 'drug_clonazepam', 'drug_ipratropium', 'drug_oxycodone', 'drug_lisinopril', 'drug_amphetamine', 'drug_methamphetamine', 'drug_zolpidem', 'drug_alcohol', 'drug_acetaminophen_hydrocodone', 'drug_acetaminophen_oxycodone', 'drug_heroin', 'drug_cocaine', 'drug_marijuana', 'drug_quetiapine', 'drug_narcotic_analgesics_nos', 'drug_benzodiazepines_nos', 'drug_drug_unknown', 'drug_poly_drugs', 'drug_antineoplastics_nos']
drug_counts = pd.DataFrame(dawn[drug_cols].sum())
drug_counts.rename(
    columns = {0: "count"}, 
    index = {
        'drug_ibuprofen': 'Ibuprofen',
        'drug_warfarin': 'Warfarin',
        'drug_acetaminophen': 'Acetaminophen',
        'drug_methadone': 'Methadone',
        'drug_amoxicillin': 'Amoxicillin',
        'drug_sulfamethoxazole_trimethoprim': 'Sulfamethoxazole-Trimethoprim',
        'drug_lorazepam': 'Lorazepam',
        'drug_alprazolam': 'Alprazolam',
        'drug_aspirin': 'Aspirin',
        'drug_clonazepam': 'Clonazepam',
        'drug_ipratropium': 'Ipratropium',
        'drug_oxycodone': 'Oxycodone',
        'drug_lisinopril': 'Lisinopril',
        'drug_amphetamine': 'Amphetamine',
        'drug_methamphetamine': 'Methamphetamine',
        'drug_zolpidem': 'Zolpidem',
        'drug_alcohol': 'Alcohol',
        'drug_acetaminophen_hydrocodone': 'Acetaminophen-Hydrocodone',
        'drug_acetaminophen_oxycodone': 'Acetaminophen-Oxycodone',
        'drug_heroin': 'Heroin',
        'drug_cocaine': 'Cocaine',
        'drug_marijuana': 'Marijuana',
        'drug_quetiapine': 'Quetiapine',
        'drug_narcotic_analgesics_nos': 'Narcotic Analgesics, NOS',
        'drug_benzodiazepines_nos': 'Benzodiazepines, NOS',
        'drug_drug_unknown': 'Unknown Drug',
        'drug_poly_drugs': 'Multiple Drugs, NOS',
        'drug_antineoplastics_nos': 'Antineoplastics, NOS'
    }, 
    inplace = True
)
drug_counts.sort_values(
    by = 'count', 
    inplace = True
)
fig, ax = plt.subplots()
ax.barh(drug_counts.index, drug_counts['count'])
plt.title(label = "Drug Occurrence Frequencies")
plt.xlabel("Number of Occurrences")
plt.show()

# Model Evaluation Metric

I use recall (i.e., true positive rate) as my model metric, as it is most appropriate for situations where false negatives are more critical than false positives. For the problem at hand, a false negative (i.e., failing to detect a negative health outcome when it occurs) is far more costly and problematic than a false positive (i.e., detecting a negative health outcome when it does not actually occur), which means that I want to maximize the true positive rate and make sure that my model can correctly detect as many true positives as possible.

# Model Fitting

## Setup for Modeling

Split the full data into training and test datasets:

In [ ]:
# create train and test splits
train, test = train_test_split(
  dawn, 
  test_size = 0.25, 
  stratify = dawn['outcome'], 
  shuffle = True, 
  random_state = 1109
)

Define predictors and outcome variable:

In [ ]:
# define predictors and outcome
x_train = train.drop(columns = ['caseid', 'outcome'])
x_test = test.drop(columns = ['caseid', 'outcome'])
y_train = train['outcome'].astype(int)
y_test = test['outcome'].astype(int)

Set up k-fold cross validation:

In [ ]:
# set up k-fold cross validation
k = 5
kfold = StratifiedKFold(
    n_splits = k, 
    shuffle = True, 
    random_state = 1109
)
print("Cross-Validation Parameter: k =", k)

Define potential parameters to test across all model types:

In [ ]:
# set parameters to test
C = np.logspace(-3, 5, 9)
n_estimators = np.linspace(100, 500, 9, dtype = int)
max_features = [5, 10, 15, 20, 25, 30, 35, 40, 45, len(x_train.columns)]
max_depth = [2, 5, 10, 15, 20]

# view parameters to test
print("Potential Values for Model Parameters:")
print("- C:", C.tolist())
print("- n_estimators:", n_estimators.tolist())
print("- max_features:", max_features)
print("- max_depth:", max_depth)

## Model 1: Logistic Regression with L1 Regularization

Cross-validate the model:

In [ ]:
# set up pipeline
lr_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy = 'most_frequent', missing_values = pd.NA)), 
    ('Scaler', StandardScaler()), 
    ('LR', LogisticRegression(solver = 'saga', penalty = 'l1', max_iter = 10000, random_state = 1109))
])

# set up parameter grid
lr_grid = {
    'LR__C': C
}

# set up cross-validation
lr_cv = RandomizedSearchCV(
    lr_pipeline, 
    lr_grid, 
    cv = kfold, 
    scoring = "recall", 
    random_state = 1109, 
    n_jobs = -1
)

# run
lr_cv.fit(x_train, y_train)
print("Done!")

View results:

In [ ]:
# view results
lr_results = pd.DataFrame(lr_cv.cv_results_)
display(lr_results.filter(items = ['rank_test_score', 'mean_test_score', 'params']).sort_values(by = 'mean_test_score', ascending = False))

Fit final model and calculate its recall:

In [ ]:
# set up pipeline for final model
best_lr_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy = 'most_frequent', missing_values = pd.NA)), 
    ('Scaler', StandardScaler()), 
    ('LR', LogisticRegression(
        C = lr_cv.best_params_['LR__C'], 
        solver = 'saga', 
        penalty = 'l1', 
        max_iter = 10000, 
        random_state = 1109)
    )
])

# fit final model
best_lr = best_lr_pipeline.fit(x_train, y_train)

# calculate recall of final model
logreg_preds = best_lr.predict(x_test)
lr_recall = round(recall_score(y_test, logreg_preds), 5)
print("Recall =", lr_recall)

I use the results of the regularized logistic regression model to perform feature selection for the other two models, although only one feature (`route_7`) qualifies for removal:

In [ ]:
# perform feature selection
lr_coefs = pd.DataFrame(best_lr.named_steps["LR"].coef_ == 0, columns = x_train.columns.tolist())
display(lr_coefs)
x_train.drop(['route_7'], axis = 'columns', inplace = True)
x_test.drop(['route_7'], axis = 'columns', inplace = True)

## Model 2: Random Forest Classifier

Cross-validate the model:

In [ ]:
# set up pipeline
rf_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy = 'most_frequent', missing_values = pd.NA)), 
    ('RFC', RandomForestClassifier(random_state = 1109))
])

# set up parameter grid
rf_grid = {
    'RFC__n_estimators': n_estimators, 
    'RFC__max_features': max_features, 
    'RFC__max_depth': max_depth
}

# set up cross-validation
rf_cv = RandomizedSearchCV(
    rf_pipeline, 
    rf_grid, 
    cv = kfold, 
    scoring = "recall", 
    random_state = 1109, 
    n_jobs = -1
)

# run
rf_cv.fit(x_train, y_train)
print("Done!")

View results:

In [ ]:
# view results
rf_results = pd.DataFrame(rf_cv.cv_results_)
display(rf_results.filter(items = ['rank_test_score', 'mean_test_score', 'params']).sort_values(by = 'mean_test_score', ascending = False))

Fit final model and calculate its recall:

In [ ]:
# set up pipeline for final model
best_rf_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy = 'most_frequent', missing_values = pd.NA)), 
    ('RFC', RandomForestClassifier(
        n_estimators = rf_cv.best_params_['RFC__n_estimators'], 
        max_features = rf_cv.best_params_['RFC__max_features'], 
        max_depth = rf_cv.best_params_['RFC__max_depth'], 
        random_state = 1109)
    )
])

# fit final model
best_rf = best_rf_pipeline.fit(x_train, y_train)

# calculate recall of final model
rf_preds = best_rf.predict(x_test)
rf_recall = round(recall_score(y_test, rf_preds), 5)
print("Recall =", rf_recall)

## Model 3: Support Vector Classifier

Cross-validate the model:

In [ ]:
# set up pipeline
svc_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy = 'most_frequent', missing_values = pd.NA)), 
    ('Scaler', StandardScaler()), 
    ('SVC', LinearSVC(random_state = 1109))
])

# set up parameter grid
svc_grid = {
    'SVC__C': C
}

# set up cross-validation
svc_cv = RandomizedSearchCV(
    svc_pipeline, 
    svc_grid, 
    cv = kfold, 
    scoring = "recall", 
    random_state = 1109, 
    n_jobs = -1
)

# run
svc_cv.fit(x_train, y_train)
print("Done!")

View results:

In [ ]:
# view results
svc_results = pd.DataFrame(svc_cv.cv_results_)
display(svc_results.filter(items = ['rank_test_score', 'mean_test_score', 'params']).sort_values(by = 'mean_test_score', ascending = False))

Fit final model and calculate its recall:

In [ ]:
# set up pipeline for final model
best_svc_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy = 'most_frequent', missing_values = pd.NA)), 
    ('SVC', LinearSVC(
        C = svc_cv.best_params_['SVC__C'], 
        random_state = 1109)
    )
])

# fit final model
best_svc = best_svc_pipeline.fit(x_train, y_train)

# calculate recall of final model
svc_preds = best_svc.predict(x_test)
svc_recall = round(recall_score(y_test, svc_preds), 5)
print("Recall =", svc_recall)

# Model Comparison

Results of final three models fit with parameters selected during cross-validation are presented below:

In [ ]:
# print results for all three models
print(
    "Logistic Regression with L1 Regularization:", "\n", 
    "- Recall = ", lr_recall, "\n", 
    "- Best Parameters: ", lr_cv.best_params_, 
    sep = ""
)
print(
    "Random Forest Classifier:", "\n", 
    "- Recall = ", rf_recall, "\n", 
    "- Best Parameters: ", rf_cv.best_params_, 
    sep = ""
)
print(
    "Support Vector Classifier:", "\n", 
    "- Recall = ", svc_recall, "\n", 
    "- Best Parameters: ", svc_cv.best_params_, 
    sep = ""
)

Within model types, recall scores were very similar between training-data and test-data models and the test-data models performed slightly better than the training-data models. However, all three model types performed relatively poorly, achieving recall scores of less than 0.5 across the board. This indicates that none of the models were overfitted, as there was not a large gap between training-data and test-data models, but rather that they were underfitted because both training-data and test-data models performed poorly.

In turn, this indicates that the models all had acceptable variance but high bias; they did not seem to adhere intensely to any patterns that were present in the training data but not the test data, but they did instead seem to be overly simple to capture enough of the detail within the data to make high-quality predictions.

With respect to interpretability and flexibility, I believe that flexibility would be the most useful characteristic to prioritize for the sort of problem these models were deployed for. The regularized logistic regression model is relatively interpretable (i.e., you can look at the coefficients to understand how it makes its decisions), but the random forest classifier and support vector classifier are less interpretable but more flexible as they are better equipped to deal with highly complex and variable data.

Given that the hypothetical intention of this project was to develop a model that could be used for prediction and risk assessment in a hospital setting, I assume that a model with high flexibility would be more useful over a model with high interpretability because of the complexity of information that dictates patient outcomes and the relative lack of need to understand why the model makes the decisions it makes. Additionally, a model deployed for this purpose would necessarily be overseen by human medical staff, meaning that any decisions it makes will be used in tandem with human decision-making which serves as a form of quality control for any unexplainable or low-quality decisions. The predictive ability of these models could potentially be remedied by including more varied and complex information (e.g., medical records data with more detail) in future model-building work, but it is uncertain how much this would improve performance and improvements would need to be empirically evaluated.